In [39]:

# 清理資料：把重複病歷號的資料刪除

# 第一步：導入必要的庫
import pandas as pd
from io import StringIO

# 第二步：定義原始數據
data = """
年齡	病歷號	性別
56 	22093763	F
78 	4090223	M
68 	3868664	M
76 	450040	M
66 	9585127	M
66 	8452992	F
53 	20363287	F
77 	21334437	F
62 	21931377	M
64 	2713565	F
47 	2862239	F
62 	2410793	M
53 	39084672	F
53 	8453410	F
61 	20622462	F
92 	8691916	M
64 	2628088	F
52 	21594390	F
58 	2867212	F
73 	3460070	M
77 	877862	F
74 	10762968	F
40 	21891613	F
63 	1989918	F
73 	20030862	M
51 	20495196	F
67 	20809255	M
102 	255954	M
53 	22052320	F
64 	2719890	M
79 	8603476	F
66 	304506	M
63 	1556084	F
37 	22249399	F
62 	22227173	M
32 	21969460	M
48 	3232056	M
69 	1760948	M
48 	1671801	F
77 	3012097	M
80 	3499065	F
69 	4064373	F
74 	3604846	F
63 	8656255	M
15 	10651070	M
52 	20050339	F
79 	2042477	F
73 	437258	M
75 	247213	F
70 	10775210	F
59 	21931555	F
81 	21375717	F
75 	20117368	M
90 	4131469	F
30 	10759148	F
57 	8444337	F
93 	3906540	M
49 	20519355	F
51 	20652739	F
78 	9549902	M
57 	21891971	M
70 	21194300	F
79 	1272745	F
73 	39184701	F
74 	1594214	F
79 	20004274	F
55 	4294534	F
56 	1199487	F
58 	1575653	F
82 	4395190	F
89 	4178659	F
69 	3868664	M
54 	20363287	F
67 	9585127	M
56 	8444337	F
63 	21931377	M
59 	4228255	M
38 	2539479	M
84 	4395190	F
64 	8792950	M
75 	10762968	F
70 	10775210	F
68 	8452992	F
47 	22324684	M
43 	4556591	F
79 	1312055	F
87 	2315853	F
56 	20555718	F
"""

# 第三步：讀取數據到 DataFrame
# 使用 StringIO 將多行字符串轉換為類似文件的對象
df = pd.read_csv(StringIO(data), sep='\s+', engine='python')

# 查看原始數據的行數
print(f"原始數據共有 {len(df)} 條記錄。")

# 第四步：去除重複的病歷號碼
# 保留第一個出現的記錄，刪除後續重複的記錄
df_cleaned = df.drop_duplicates(subset='病歷號', keep='first')

# 查看清理後數據的行數
print(f"清理後數據共有 {len(df_cleaned)} 條記錄。")

# 第五步：顯示清理後的數據
print("\n清理後的數據如下：")
print(df_cleaned)

原始數據共有 88 條記錄。
清理後數據共有 79 條記錄。

清理後的數據如下：
    年齡       病歷號 性別
0   56  22093763  F
1   78   4090223  M
2   68   3868664  M
3   76    450040  M
4   66   9585127  M
..  ..       ... ..
83  47  22324684  M
84  43   4556591  F
85  79   1312055  F
86  87   2315853  F
87  56  20555718  F

[79 rows x 3 columns]


In [43]:
# 主要計算過程：設定平均數、標準差、性別比的檢驗門檻，隨機抽樣迭代一萬次。並把符合的抽樣結果打印出來


# 第一步：導入必要的庫
import pandas as pd
import numpy as np

# 假設 df_cleaned 已經存在，並包含 '性別' 和 '年齡' 欄位

# 第二步：分割性別數據
males = df_cleaned[df_cleaned['性別'] == 'M']
females = df_cleaned[df_cleaned['性別'] == 'F']

num_males = len(males)
num_females = len(females)

print(f"男性共有 {num_males} 人，女性共有 {num_females} 人。")

# 檢查是否有足夠的男性和女性進行抽樣
required_males = 17
required_females = 36

if num_males < required_males:
    raise ValueError(f"男性數量不足，無法抽取 {required_males} 人。當前男性數量：{num_males}")
if num_females < required_females:
    raise ValueError(f"女性數量不足，無法抽取 {required_females} 人。當前女性數量：{num_females}")

# 第三步：隨機抽樣和篩選
iterations = 10000
matching_subsets = []  # 用於保存符合條件的子集
count_matches = 0

# 將性別分開並獲取索引
males_indices = males.index.values
females_indices = females.index.values

for i in range(iterations):
    # 隨機抽取17名男性和36名女性的索引
    sampled_males_indices = np.random.choice(males_indices, size=required_males, replace=False)
    sampled_females_indices = np.random.choice(females_indices, size=required_females, replace=False)

    # 根據索引選取年齡
    sampled_males_ages = males.loc[sampled_males_indices]['年齡'].values
    sampled_females_ages = females.loc[sampled_females_indices]['年齡'].values

    # 合併年齡
    subset_ages = np.concatenate([sampled_males_ages, sampled_females_ages])

    # 計算平均值和標準差
    mean_age = subset_ages.mean()
    std_age = subset_ages.std()

    # 檢查條件（根據新的範圍調整）
    if 63.9 <= mean_age <= 64.0 and 15.4 <= std_age <= 15.5:
        count_matches += 1
        # 保存符合條件的子集詳細資訊
        matching_subset = pd.concat([
            males.loc[sampled_males_indices],
            females.loc[sampled_females_indices]
        ])
        matching_subsets.append({
            'iteration': i+1,
            'mean_age': mean_age,
            'std_age': std_age,
            'subset': matching_subset
        })

    # 每1000次迭代打印一次進度
    if (i+1) % 1000 == 0:
        print(f"已完成 {i+1} 次抽樣。")

# 第四步：結果統計與打印符合條件的子集
print(f"\n在 {iterations} 次抽樣中，有 {count_matches} 次符合條件。")
percentage = (count_matches / iterations) * 100
print(f"符合條件的比例為 {percentage:.2f}%。\n")

# 打印所有符合條件的子集詳情
if count_matches > 0:
    print("符合條件的子集詳情如下：\n")
    for subset in matching_subsets:
        print(f"迭代次數: {subset['iteration']}")
        print(f"平均年齡: {subset['mean_age']:.2f}")
        print(f"標準差: {subset['std_age']:.2f}")
        print("抽取的子集資料：")
        print(subset['subset'].to_string(index=False))
        print("\n" + "-"*50 + "\n")
else:
    print("在所有抽樣中未找到符合條件的子集。")

男性共有 29 人，女性共有 50 人。
已完成 1000 次抽樣。
已完成 2000 次抽樣。
已完成 3000 次抽樣。
已完成 4000 次抽樣。
已完成 5000 次抽樣。
已完成 6000 次抽樣。
已完成 7000 次抽樣。
已完成 8000 次抽樣。
已完成 9000 次抽樣。
已完成 10000 次抽樣。

在 10000 次抽樣中，有 8 次符合條件。
符合條件的比例為 0.08%。

符合條件的子集詳情如下：

迭代次數: 1576
平均年齡: 64.00
標準差: 15.49
抽取的子集資料：
 年齡      病歷號 性別
 73 20030862  M
 62  2410793  M
 76   450040  M
 48  3232056  M
 78  4090223  M
 62 22227173  M
 32 21969460  M
 73  3460070  M
 15 10651070  M
 93  3906540  M
102   255954  M
 78  9549902  M
 64  2719890  M
 62 21931377  M
 73   437258  M
 75 20117368  M
 59  4228255  M
 74 10762968  F
 63  1989918  F
 79 20004274  F
 57  8444337  F
 58  1575653  F
 63  1556084  F
 64  2713565  F
 90  4131469  F
 53  8453410  F
 59 21931555  F
 52 20050339  F
 56  1199487  F
 55  4294534  F
 43  4556591  F
 79  1272745  F
 49 20519355  F
 74  3604846  F
 53 20363287  F
 74  1594214  F
 73 39184701  F
 80  3499065  F
 77 21334437  F
 81 21375717  F
 64  2628088  F
 75   247213  F
 58  2867212  F
 69  4064373  F
 51 20652739  F
 30